# Messages

In [20]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
from pathlib import Path

from langchain_core.messages import HumanMessage
from rich import print as rprint
import asyncio
import time
import base64

# 从.env文件中加载环境变量
load_dotenv(Path('../.env'), override=True)
DASHSCOPE_MODEL_NAME = "qwen3.7-plus"
DASHSCOPE_MODEL_PROVIDER = os.getenv("DASHSCOPE_MODEL_PROVIDER")
DASHSCOPE_API_KEY = os.getenv("DASHSCOPE_API_KEY")
DASHSCOPE_BASE_URL = os.getenv("DASHSCOPE_BASE_URL")

model = init_chat_model(
    model=DASHSCOPE_MODEL_NAME,
    model_provider=DASHSCOPE_MODEL_PROVIDER,
    api_key=DASHSCOPE_API_KEY,
    base_url=DASHSCOPE_BASE_URL
)


def encode_image(img_path, img_type='jpeg'):
    """将一张本地图片转换成 Base64 编码的 Data URI 字符串,方便在文本中嵌入图片数据"""
    with open(img_path, "rb") as img_file:
        return f"data:image/{img_type};base64,{base64.b64encode(img_file.read()).decode("utf-8")}"

## 传统content传入多模态数据
- 部分模型无法识别

In [21]:
img_base64 = encode_image('./image_test.png', img_type='png')

result_stream = model.stream([
    HumanMessage(content=[
        {'type': 'text', 'text': '这张图片里的是什么？'},
        {'type': 'image_url', 'image_url': img_base64},
    ]),
])

for resp in result_stream:
    print(resp.content, end='', flush=True)

这张图片里展示的是 **雅诗兰黛（Estée Lauder）的 Double Wear 持妆粉底液**（Double Wear Stay-in-Place Makeup）。

从图片特征可以看出：
1.  **瓶身设计**：经典的方形玻璃瓶身，里面装着米色/肤色的粉底液。
2.  **标签**：瓶身中间有金色的标签，上面隐约可以看到 "Estée Lauder" 和 "Double Wear" 的字样。
3.  **瓶盖**：透明的方形瓶盖，带有金色的泵头设计。

这款粉底液通常被称为“DW粉底液”，以其高遮瑕力和持久持妆效果而闻名。

## 老式content_blocks多模态输入
- 统一多模态调用输入。传统方式，将多模态数据和文本消息放入到content中，但有些模型(像ChatGPT, Claude)无法识别，而content_blocks是新版本统一方法

In [25]:
img_base64 = encode_image('./image_test.png', img_type='png')

result_stream = model.stream([
    HumanMessage(content_blocks=[
        {'type': 'text', 'text': '这张图片里的是什么？'},
        {'type': 'image', 'url': img_base64},
    ]),
])

for resp in result_stream:
    print(resp.content, end='', flush=True)

这张图片展示的是一瓶 **雅诗兰黛（Estée Lauder）的 Double Wear 持妆粉底液**（通常被大家简称为“DW粉底液”）。

从图片特征可以看出来：
1.  **瓶身设计**：它有着标志性的长方形透明玻璃瓶身。
2.  **标签**：瓶身中间有一个金色的标签，上面隐约可以看到 "Estée Lauder" 和 "Double Wear" 的字样。
3.  **内容物**：瓶内装着米色/肤色的液体，这是粉底液的颜色。
4.  **瓶盖**：顶部是透明带有金色内管的按压式泵头设计（或者是这种风格的盖子）。

这是一款非常著名的主打持久、遮瑕的粉底液产品。

## 新版content_blocks

In [27]:
from langchain_core.messages.content import create_text_block, create_image_block

img_base64 = encode_image('./image_test.png', img_type='png')

result_stream = model.stream([
    HumanMessage(content_blocks=[
        create_text_block("这张图片是什么？"),
        # 这里编码后前缀是data:image，这是url，不是纯base64，所以关键字参数用url，不用base64
        create_image_block(url=img_base64, mime_type='image/png'),
    ]),
])

for resp in result_stream:
    print(resp.content, end='', flush=True)

这张图片展示的是 **雅诗兰黛（Estée Lauder）的 Double Wear 持妆粉底液**（Double Wear Stay-in-Place Makeup）。

从图片特征来看：
1.  **瓶身设计**：采用了标志性的方形透明玻璃瓶身，可以看到里面米色/肤色的粉底液液体。
2.  **标签**：瓶身中间有金色的标签，上面印有 "Estée Lauder" 品牌名和 "Double Wear" 产品名。
3.  **外观**：这是非常经典的一款粉底液，通常被称为“DW粉底液”，以持妆时间长、遮瑕力高而闻名。

图片中的光影效果（暖色调背景）是为了展示产品的质感和色号。